# 🧹 Nettoyage d'un export de production — `ventes_corrompu.csv`

**Objectif** : transformer un export sale (doublons, types incohérents, manquants, aberrations,
libellés hétérogènes) en un jeu propre, prêt à l'analyse. On documente **avant / après**.

> **Analogie** — nettoyer une donnée, c'est **préparer les légumes avant de cuisiner** : on lave,
> on retire les parties abîmées, on calibre. Sauter cette étape gâche tout le plat (l'analyse).

In [ ]:
import pandas as pd
import numpy as np

DATA = "../../99-Brief/Data-Analyst/data"
brut = pd.read_csv(f"{DATA}/ventes_corrompu.csv")
print("Forme brute :", brut.shape)
brut.head()

In [ ]:
brut.info()

## 1. Diagnostic
On liste les problèmes AVANT de corriger (règle d'or : diagnostiquer, puis traiter).

In [ ]:
print("Doublons :", brut.duplicated().sum())
print("\nManquants par colonne :\n", brut.isna().sum())
print("\nExemples de dates :", brut["date"].dropna().unique()[:5])
print("\nVilles (casse hétérogène ?) :", brut["ville"].dropna().unique()[:10])

## 2. Doublons

In [ ]:
df = brut.drop_duplicates().copy()
print("Lignes après dédoublonnage :", len(df))

## 3. Types : dates et nombres
Les dates arrivent en formats mélangés (`11/10/2024`…) ; on les normalise. `errors="coerce"`
transforme l'invalide en `NaT`/`NaN` au lieu de planter.

In [ ]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
for col in ["quantite","prix_unitaire","remise","montant","marge"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.dtypes

## 4. Libellés : harmoniser la casse et les espaces

In [ ]:
for col in ["ville","type","categorie","produit"]:
    df[col] = df[col].astype(str).str.strip().str.title()
df["ville"].unique()[:10]

## 5. Valeurs manquantes
On distingue : lignes inutilisables (clé/mesure critique manquante) vs manquants tolérables.

In [ ]:
avant = len(df)
df = df.dropna(subset=["date","montant"])   # sans date ni montant, la ligne est inexploitable
df["remise"] = df["remise"].fillna(0)         # une remise absente = pas de remise
print(f"Lignes supprimées (date/montant manquants) : {avant - len(df)}")

## 6. Aberrations (outliers) — méthode de l'IQR
Un montant négatif ou démesuré est suspect. On borne via l'écart interquartile (IQR).

In [ ]:
q1, q3 = df["montant"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5*iqr, q3 + 1.5*iqr
suspects = df[(df["montant"] < borne_basse) | (df["montant"] > borne_haute)]
print(f"Bornes plausibles : [{borne_basse:.0f} ; {borne_haute:.0f}] €")
print(f"Montants suspects : {len(suspects)}")
df_clean = df[(df["montant"] >= max(0, borne_basse)) & (df["montant"] <= borne_haute)].copy()

## 7. Bilan avant / après

In [ ]:
print(f"Avant : {brut.shape[0]} lignes")
print(f"Après : {df_clean.shape[0]} lignes propres")
print(f"Taux de rétention : {len(df_clean)/len(brut):.1%}")
df_clean.describe()

## 🎯 À toi de jouer
Combien de lignes avaient une **quantité négative ou nulle** dans le fichier dédoublonné `df` ?

In [ ]:
# Écris ta réponse ici :


<details><summary>💡 Corrigé</summary>

```python
(df['quantite'] <= 0).sum()
```
</details>

## ✅ À retenir
- **Diagnostiquer d'abord** (doublons, types, manquants, aberrations), traiter ensuite.
- `pd.to_datetime(..., errors="coerce")` et `pd.to_numeric(..., errors="coerce")` sécurisent les conversions.
- Harmoniser la casse (`str.title()`) évite les faux doublons (« lille » ≠ « Lille »).
- Les outliers se repèrent avec l'**IQR** ; on documente toujours le **taux de rétention**.